실습 3: 데이터 로드 및 기본 탐색
================================
CSIC 2010 HTTP 데이터셋(csic2010_requests.csv)을 로드하고
HTTP 요청의 구조와 정상/공격 분포를 파악합니다.

실행 방법: JupyterLab에서 이 노트북을 열고 셀을 위에서부터 Shift+Enter로 순차 실행
사전 준비: python download_csic2010.py 로 csic2010_requests.csv 생성


In [4]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from urllib.parse import unquote

# 시각화 스타일 설정 (다크모드/라이트모드 호환)
plt.style.use('seaborn-v0_8-muted') 
print("✅ 패키지 로드 완료")

✅ 패키지 로드 완료


In [5]:
try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    SCRIPT_DIR = os.getcwd()

path_61k = os.path.join(SCRIPT_DIR, "csic2010_requests_61000.csv")
path_10k = os.path.join(SCRIPT_DIR, "csic2010_requests_10000.csv")

# 데이터프레임 생성
df_61k = pd.read_csv(path_61k)
df_10k = pd.read_csv(path_10k)

print(f"✅ 데이터 로드 성공: 61k({len(df_61k):,}건), 10k({len(df_10k):,}건)")

✅ 데이터 로드 성공: 61k(61,065건), 10k(10,000건)


In [6]:
summary = pd.DataFrame({
    '61k 데이터셋': [f"{df_61k.shape[0]:,} 행", f"{df_61k.shape[1]} 열", f"{df_61k.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB"],
    '10k 데이터셋': [f"{df_10k.shape[0]:,} 행", f"{df_10k.shape[1]} 열", f"{df_10k.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB"]
}, index=['샘플 수', '변수 수', '메모리 점유'])

display(summary)

,61k 데이터셋,10k 데이터셋
샘플 수,"61,065 행","10,000 행"
변수 수,18 열,18 열
메모리 점유,33.9 MB,5.6 MB


In [7]:
def check_label(df):
    return df["label"].value_counts(normalize=True).mul(100).round(1)

label_comp = pd.DataFrame({
    '61k 비율(%)': check_label(df_61k),
    '10k 비율(%)': check_label(df_10k)
})

print("📊 데이터 라벨 구성비 비교")
display(label_comp)

📊 데이터 라벨 구성비 비교


,61k 비율(%),10k 비율(%)
label,,
Normal,59.0,59.0
Anomalous,41.0,41.0


In [8]:
method_61k = df_61k["method"].value_counts(normalize=True).mul(100)
method_10k = df_10k["method"].value_counts(normalize=True).mul(100)

method_comp = pd.DataFrame({
    '61k 점유율': method_61k,
    '10k 점유율': method_10k
}).fillna(0).round(1)

print("📈 HTTP 메서드 분포 비교")
display(method_comp)

📈 HTTP 메서드 분포 비교


,61k 점유율,10k 점유율
method,,
GET,70.6,69.8
POST,28.8,29.4
PUT,0.7,0.7


In [9]:
df_61k["url_len"] = df_61k["url"].str.len()
df_10k["url_len"] = df_10k["url"].str.len()

stats_61k = df_61k.groupby("label")["url_len"].agg(['mean', 'median', 'max']).round(1)
stats_10k = df_10k.groupby("label")["url_len"].agg(['mean', 'median', 'max']).round(1)

print("📐 URL 길이 수치 통계 비교 (위: 61k / 아래: 10k)")
display(stats_61k)
display(stats_10k)

📐 URL 길이 수치 통계 비교 (위: 61k / 아래: 10k)


,mean,median,max
label,,,
Anomalous,77.2,36.0,865.0
Normal,49.0,28.5,337.0


,mean,median,max
label,,,
Anomalous,77.5,33.0,865.0
Normal,50.5,29.0,332.0


In [10]:
def get_ct(df):
    return df["content_type"].fillna("GET/기타").value_counts(normalize=True).mul(100)

ct_comp = pd.DataFrame({
    '61k 데이터': get_ct(df_61k),
    '10k 데이터': get_ct(df_10k)
}).fillna(0).round(2)

print("📝 Content-Type 분포 현황 (%)")
display(ct_comp)

📝 Content-Type 분포 현황 (%)


,61k 데이터,10k 데이터
content_type,,
GET/기타,70.56,69.84
application/x-www-form-urlencoded,29.44,30.16


In [11]:
keywords = ['select', 'union', '<script', 'alert', '../']

def check_density(df):
    full_text = df["url"].fillna("") + " " + df["body"].fillna("")
    attack_text = full_text[df["label"] == "Anomalous"]
    return pd.Series({kw: (attack_text.str.contains(kw, case=False).mean() * 100).round(1) for kw in keywords})

kw_comp = pd.DataFrame({
    '61k 공격 내 검출률(%)': check_density(df_61k),
    '10k 공격 내 검출률(%)': check_density(df_10k)
})

print("🎯 핵심 시그니처 검출 밀도 비교")
display(kw_comp)

🎯 핵심 시그니처 검출 밀도 비교


,61k 공격 내 검출률(%),10k 공격 내 검출률(%)
select,0.8,0.9
union,0.0,0.0
<script,0.0,0.0
alert,4.6,4.7
../,95.9,95.8
